# Phase 6: Audio Preprocessing Pipeline

This notebook demonstrates and benchmarks the unified audio preprocessing pipeline (resampling, mono conversion, VAD silence trimming, QA scoring, and loudness normalization) enforcing strict fold-local statistics (`RC-08`).

In [ ]:
# Step 1: Environment & Dependency Check
import sys
import json
from pathlib import Path
import numpy as np

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from model.src.preprocessing.pipeline import preprocess_recording
from model.src.preprocessing.normalize import compute_fold_normalization_stats
from model.src.preprocessing.quality_check import compute_quality_score

print("✓ Preprocessing modules successfully imported.")

In [ ]:
# Step 2: Load Audio Configuration & Split Manifest
import yaml

config_path = Path("model/configs/audio_config.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    audio_config = yaml.safe_load(f)

split_manifest_path = Path("model/artifacts/split_manifest_v1.0-20260823.json")
with open(split_manifest_path, "r", encoding="utf-8") as f:
    split_manifest = json.load(f)

print("=== Audio Configuration ===")
print(json.dumps(audio_config, indent=2))
print(f"\n✓ Loaded Split Manifest: {split_manifest['dataset_name']} ({split_manifest['dataset_version']})")
print(f"  - Train Speakers: {len(split_manifest['splits']['train']['speakers'])}")
  print(f"  - Val Speakers:   {len(split_manifest['splits']['val']['speakers'])}")
  print(f"  - Test Speakers:  {len(split_manifest['splits']['test']['speakers'])}")

In [ ]:
# Step 3: Fold-Local Normalization Statistics Fitting (RC-08)
print("=== Fitting Fold-Local Statistics on Train Split Only ===")

# Synthesize/load training audio representations
sr = audio_config.get("target_sample_rate", 16000)
train_speakers = split_manifest["splits"]["train"]["speakers"]

# Create sample fixtures for train split to simulate fold fitting
train_audio_samples = []
for spk in train_speakers:
    t = np.linspace(0, 3.0, int(sr * 3.0), endpoint=False)
    tone = (0.5 * np.sin(2 * np.pi * 220.0 * t)).astype(np.float32)
    train_audio_samples.append(tone)

# Fit stats exclusively on training fold
fold_stats = compute_fold_normalization_stats(train_audio_samples, fold_id="fold_0")
print("✓ Fitted Fold Statistics (Fold 0):")
print(json.dumps(fold_stats, indent=2))

In [ ]:
# Step 4: Run Preprocessing on Train, Val, and Test Audio
print("=== Running Preprocessing Pipeline Across Partitions ===")

partitions = ["train", "val", "test"]
results_summary = []

for part in partitions:
    spks = split_manifest["splits"][part]["speakers"]
    for spk in spks:
        # Generate test synthetic recording for speaker
        t = np.linspace(0, 4.0, int(sr * 4.0), endpoint=False)
        # Add silence pad + tone
        raw_sig = np.concatenate([
            np.zeros(int(sr * 0.5), dtype=np.float32),
            (0.6 * np.sin(2 * np.pi * 350.0 * t)).astype(np.float32),
            np.zeros(int(sr * 0.5), dtype=np.float32)
        ])
        
        # Run unified preprocessing pipeline
        proc = preprocess_recording(
            raw_sig,
            orig_sr=sr,
            config=audio_config,
            fold_stats=fold_stats
        )
        
        q = proc["quality"]
        results_summary.append({
            "split": part,
            "speaker_id": spk,
            "processed_duration": proc["duration_sec"],
            "quality_score": q["audio_quality_score"],
            "snr_db": q["estimated_snr_db"],
            "clipping_ratio": q["clipping_ratio"],
            "is_acceptable": proc["is_acceptable"]
        })

import pandas as pd
df_res = pd.DataFrame(results_summary)
print("\n--- Preprocessing QA Summary Table ---")
print(df_res.to_string(index=False))

# Assert all clean audio fixtures passed QA
assert df_res['is_acceptable'].all(), "FAIL: Some valid recordings failed QA checks!"
print("\n✓ All partition samples passed audio quality verification.")

In [ ]:
# Step 5: Verify Fold-Local Isolation (Constraint RC-08)
print("=== Verifying Fold-Local Isolation (RC-08) ===")
assert fold_stats["fitted_sample_count"] == len(train_speakers), "Leakage error: sample count mismatch!"
assert "val" not in str(fold_stats), "Validation data leaked into fold stats!"
assert "test" not in str(fold_stats), "Test data leaked into fold stats!"
print("✓ Confirmed: Fold normalization statistics derived exclusively from training partition.")
print("\n✓ Phase 6 Complete: Shared preprocessing pipeline validated and ready for model backbones.")